# V-JEPA 2.1 feature extraction demo

This notebook extracts image and video features with the smallest supported model (`vjepa2_1_vit_base_384`) using both the PyTorch reference backend and the MLX port.

It intentionally loads only one backend at a time to keep peak memory lower. For each modality it:

- preprocesses the input once
- runs PyTorch and releases the backend
- runs MLX and releases the backend
- prints the first 10 and last 10 flattened feature values from both outputs
- prints comparison metrics

Prerequisites:

```bash
uv sync --extra dev --extra notebook
./scripts/clone_reference_vjepa2.sh
uv run vjepa2-1 download-assets --model vjepa2_1_vit_base_384
uv run jupyter notebook
```

In [ ]:
from pathlib import Path
import gc

import mlx.core as mx
import numpy as np
import torch

from vjepa2_1_mlx.backends.mlx import MLXBackend
from vjepa2_1_mlx.backends.pytorch import PyTorchBackend
from vjepa2_1_mlx.compare import compare_features
from vjepa2_1_mlx.constants import MODEL_SPECS
from vjepa2_1_mlx.preprocessing import load_input_tensor


def find_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate.resolve()
    raise RuntimeError("Could not locate repo root from the current working directory")


REPO_ROOT = find_repo_root()
MODEL_NAME = "vjepa2_1_vit_base_384"
SPEC = MODEL_SPECS[MODEL_NAME]
IMAGE_PATH = REPO_ROOT / ".artifacts" / "samples" / "sample_image.png"
VIDEO_PATH = REPO_ROOT / ".artifacts" / "samples" / "sample_video.mp4"

for path in (IMAGE_PATH, VIDEO_PATH):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing sample asset: {path}. Run `uv run vjepa2-1 download-assets --model {MODEL_NAME}` first."
        )

print(f"Repo root: {REPO_ROOT}")
print(f"Model: {MODEL_NAME}")

In [ ]:
def run_pytorch_features(input_tensor: np.ndarray) -> np.ndarray:
    backend = PyTorchBackend(MODEL_NAME, device="cpu")
    try:
        features = backend.infer(input_tensor)
    finally:
        del backend
        gc.collect()
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()
    return features


def run_mlx_features(input_tensor: np.ndarray) -> np.ndarray:
    backend = MLXBackend(MODEL_NAME)
    try:
        features = backend.infer(input_tensor)
    finally:
        del backend
        gc.collect()
        mx.clear_cache()
    return features


def first_last_10(features: np.ndarray):
    flat = features.reshape(-1)
    return flat[:10], flat[-10:]


def compare_backends(label: str, input_tensor: np.ndarray):
    print(f"=== {label} ===")
    print(f"Input shape: {input_tensor.shape}")
    pytorch_features = run_pytorch_features(input_tensor)
    mlx_features = run_mlx_features(input_tensor)
    metrics = compare_features(pytorch_features, mlx_features)

    pt_first, pt_last = first_last_10(pytorch_features)
    mlx_first, mlx_last = first_last_10(mlx_features)

    print("PyTorch output shape:", pytorch_features.shape)
    print("MLX output shape:", mlx_features.shape)
    print("PyTorch first 10:", np.array2string(pt_first, precision=7, suppress_small=False))
    print("MLX first 10:    ", np.array2string(mlx_first, precision=7, suppress_small=False))
    print("PyTorch last 10: ", np.array2string(pt_last, precision=7, suppress_small=False))
    print("MLX last 10:     ", np.array2string(mlx_last, precision=7, suppress_small=False))
    print("Metrics:", metrics)

    return {
        "pytorch_features": pytorch_features,
        "mlx_features": mlx_features,
        "metrics": metrics,
    }

In [ ]:
image_tensor = load_input_tensor(
    input_path=IMAGE_PATH,
    input_type="image",
    image_size=SPEC["img_size"],
    num_frames=SPEC["num_frames"],
)
image_result = compare_backends("Image features", image_tensor)

In [ ]:
video_tensor = load_input_tensor(
    input_path=VIDEO_PATH,
    input_type="video",
    image_size=SPEC["img_size"],
    num_frames=SPEC["num_frames"],
)
video_result = compare_backends("Video features", video_tensor)